In [18]:
import numpy as np
import pandas as pd
import wfdb
from pathlib import Path
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt, find_peaks
from textformat import TextFormat
from textformat.progress import DownloadProgressBar
import glob
import jax
import jax.numpy as jnp
from jax import jit, vmap
from functools import partial

np.random.seed(42)
ROOT = Path("../../").resolve()
DB_DIR = ROOT / "db"
MIMICIV_DIR = DB_DIR / "mimiciv"
MIMICIV_ECG_DIR = DB_DIR / "mimiciv_ecg"
PROCESSED_DIR = DB_DIR / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(TextFormat.style(("MIMIC‑IV:" + str(MIMICIV_DIR)), TextFormat.COLORS["magenta"]))
print(TextFormat.style(("MIMIC‑IV‑ECG:" + str(MIMICIV_ECG_DIR)), TextFormat.COLORS["magenta"]))
print(TextFormat.style(("Processed:" + str(PROCESSED_DIR)), TextFormat.COLORS["magenta"]))

MIMIC‑IV:/mnt/a/CVD-Predict/db/mimiciv
MIMIC‑IV‑ECG:/mnt/a/CVD-Predict/db/mimiciv_ecg
Processed:/mnt/a/CVD-Predict/db/processed


In [19]:
patients = pd.read_csv(list(MIMICIV_DIR.rglob("patients.csv"))[0])
admissions = pd.read_csv(list(MIMICIV_DIR.rglob("admissions.csv"))[0])
diagnoses = pd.read_csv(list(MIMICIV_DIR.rglob("diagnoses_icd.csv"))[0])

icustays_path = list(MIMICIV_DIR.rglob("icustays.csv"))
icustays = pd.read_csv(icustays_path[0]) if icustays_path else None

print(TextFormat.style("Loaded data files", TextFormat.COLORS["green"]))

patients.columns = patients.columns.str.lower()
admissions.columns = admissions.columns.str.lower()
diagnoses.columns = diagnoses.columns.str.lower()
if icustays is not None:
    icustays.columns = icustays.columns.str.lower()

stroke_codes = ["I60","I61","I62","I63","I64"]
diagnoses["stroke"] = diagnoses["icd_code"].astype(str).str.startswith(tuple(stroke_codes))

stroke_labels = (
    diagnoses.groupby("subject_id")["stroke"]
    .max()
    .reset_index()
)

print(TextFormat.style(("Stroke rate:" + str(stroke_labels.stroke.mean())), TextFormat.COLORS["blue"]))

if icustays is not None:
    icu_summary = (
        icustays.groupby("subject_id")
        .agg(icu_los=("los","mean"))
        .reset_index()
    )
    icu_summary = icu_summary[icu_summary["icu_los"] < 2]
    keep_subjects = set(icu_summary.subject_id)
else:
    keep_subjects = set(stroke_labels.subject_id)

print(TextFormat.style(("Subjects kept:" + str(len(keep_subjects))), TextFormat.COLORS["blue"]))

Loaded data files
Stroke rate:0.03258528109059478
Subjects kept:32033


In [30]:
record_list = MIMICIV_ECG_DIR / "record_list.csv"
ecg_index = pd.read_csv(record_list)
ecg_index.columns = ecg_index.columns.str.lower()
ecg_index = ecg_index[ecg_index.subject_id.isin(keep_subjects)]
print("ECG records:", len(ecg_index))

FS = 250
LOW = 5
HIGH = 15

b, a = butter(1, [LOW/(FS/2), HIGH/(FS/2)], btype='band')

def bandpass_filter_fast(ecg: np.ndarray) -> np.ndarray:
    return filtfilt(b, a, ecg).astype(np.float32)

@jit
def fast_rpeak_detector(x):
    x2 = x * x

    win_size = 37
    kernel = jnp.ones((win_size,), dtype=jnp.float32) / win_size
    mwa = jnp.convolve(x2, kernel, mode='same')

    is_peak = (
        (mwa > jnp.roll(mwa, 1)) &
        (mwa > jnp.roll(mwa, -1))
    )

    min_dist = 62

    def scan_fn(carry, inputs):
        last_idx = carry
        idx, peak_flag = inputs

        keep = peak_flag & (idx - last_idx >= min_dist)

        new_last = jnp.where(keep, idx, last_idx)
        out = jnp.where(keep, idx, -1)

        return new_last, out

    idxs = jnp.arange(x.shape[0])

    _, peaks = jax.lax.scan(
        scan_fn,
        jnp.array(-100000, dtype=jnp.int32),
        (idxs, is_peak)
    )

    return peaks

@jit
def fast_hr_from_rpeaks(rpeaks):
    valid = rpeaks >= 0
    rpeaks = jnp.where(valid, rpeaks, 0)

    rr = jnp.diff(rpeaks) / 250.0

    rr_valid = (rr > 0.3) & (rr < 2.0)
    rr = jnp.where(rr_valid, rr, jnp.nan)

    mean_rr = jnp.nanmean(rr)

    hr = 60.0 / mean_rr
    hr = jnp.where((hr >= 30) & (hr <= 220), hr, jnp.nan)

    return hr

SEQ_LEN = 1440

def extract_hr_fast(ecg_np: np.ndarray) -> jnp.ndarray:
    filtered = bandpass_filter_fast(ecg_np)

    x = jnp.asarray(filtered)

    rpeaks = fast_rpeak_detector(x)
    hr = fast_hr_from_rpeaks(rpeaks)

    hr_seq = jnp.full((SEQ_LEN,), hr, dtype=jnp.float32)

    return hr_seq

ECG records: 170806


In [32]:
windows = []

def process_ecgs_with_progress(windows):
    total = len(ecg_index.head(5000))
    processed = 0

    progress = DownloadProgressBar(
        total=total,
        prefix='Processing ECGs: '
    )

    for _, row in ecg_index.head(5000).iterrows():
        processed += 1
        success = False

        try:
            path = MIMICIV_ECG_DIR / row["path"]
            record = wfdb.rdrecord(str(path))
            ecg = record.p_signal[:, 0].astype("float32")

            hr_fixed = extract_hr_fast(ecg)
            if hr_fixed is not None and len(hr_fixed) >= 10:
                windows.append({
                    "subject_id": row.subject_id,
                    "hr": hr_fixed,
                })
                success = True

        except Exception as ex:
            print(f"Error: {path}, {ex}")

        finally:
            progress.update(processed)

    print("\n✅ Processing complete.")
    return windows

windows = process_ecgs_with_progress(
    windows=windows
)
windows = pd.DataFrame(windows)
print("Windows built:", len(windows))

Processing ECGs:  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.0% • 4.9 KiB/4.9 KiB • 4.7 B/s • 0:0005
✅ Processing complete.
Windows built: 5000


In [48]:
del pd
import pandas as pd
clean_hr = []
import joblib
for x in dataset["hr"].values:
    x = np.asarray(x, dtype=np.float32)

    x = x.tolist()

    clean_hr.append(x)

dataset["hr"] = clean_hr

dataset["stroke"] = dataset["stroke"].astype(np.int8)
dataset["subject_id"] = dataset["subject_id"].astype(np.int32)
print(dataset.info())

joblib.dump(dataset, PROCESSED_DIR / "stroke_wearable_dataset.pkl")
dataset.to_parquet(
    PROCESSED_DIR / "stroke_wearable_dataset.parquet",
    engine="fastparquet"
)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   subject_id  5000 non-null   int32 
 1   hr          5000 non-null   object
 2   stroke      5000 non-null   int8  
dtypes: int32(1), int8(1), object(1)
memory usage: 63.6+ KB
None
